# §1.2 draft — XML structure and profile

Copy these cells into `wip_echo.ipynb` under the `### 1.2 XML structure and profile` heading.
They assume `XML_PATH`, `DICT_PATH` (§0) and `check_names` (§1.1) already exist.

Every block answers the same four questions, in this order — **why I need it · what it is ·
how I get it · why the others care**. That is the house annotation rule for this project: a
cell nobody can answer those four about does not ship.

### 1.2 XML structure and profile

**Why I need it.** Task 1 requires the grain, nesting and key structure of *both* sources. The
XML is the harder half: it carries two tables the JSON does not have (`products`, and a
warehouse directory), and every value arrives as text.

**What it is.** The same contract as §1.1 — `parse_xml()` returns flat DataFrames keyed by
output-table name, canonical column names, source-native values (DEC-011).

**How I get it.** `xml.etree.ElementTree`, a structured parser. The spec forbids treating either
document as plain text for regex, so no string scraping anywhere in this section.

**Why the others care.** This is the only source of `products`, which Jasmine cannot build
without it, and the only place the `DD/MM/YYYY`, `AUD 2,765.47`, `Y/N` and `10%` formats appear
— every one of which becomes a normalisation rule she writes and Yandu validates.

#### Structure survey

**Why I need it.** Same reason as the JSON: state the shape before flattening it, so the
flattening is a consequence of evidence rather than of my assumptions about the file.

**What it is.** The root element and its attributes, the five collections beneath it and their
sizes, and the internal shape of one `Order`.

**How I get it.** `.findall()` with a path, then count. No DataFrames yet — this is inspection.

**Why the others care.** The counts here are the denominator for every row-flow check in §6.
If 2,818 orders go in and 2,750 come out, Yandu has to be able to point at this cell.

In [ ]:
root = ET.parse(XML_PATH).getroot()

print("Root element   :", root.tag)
print("Root attributes:", root.attrib)          # groupAlias / sourceSystem / period

for child in root:
    print(f"  {child.tag:20s} {len(list(child)):>6,} children")

# One order, to show the nesting. Compare with the JSON: same three parts, different names.
order = root.find("Orders/Order")
print("\nAn Order contains:", [part.tag for part in order])
print("  Header        ->", len(list(order.find("Header"))), "fields  (grain: one row per order)")
print("  Shopping_Cart ->", len(order.findall("Shopping_Cart/Item")), "Item(s) (grain: one row per order item)")
print("  Delivery      ->", len(list(order.find("Delivery"))), "fields  (grain: one row per order)")

# WarehouseDirectory has no matching output table. Noted now so it is not silently dropped.
print("\nWarehouse fields:", [f.tag for f in root.find("WarehouseDirectory/Warehouse")])

#### Naming rule and the parser

**Why I need it.** The XML uses `Title_Case_With_Underscores`; the data dictionary uses
`snake_case`. A rule is needed, and by DEC-012 it must be tested rather than assumed.

**What it is.** `Order_ID` → `order_id` is just `.lower()` — a far simpler rule than the JSON's
camelCase regex, because the underscores are already there. `element_to_record` turns one
element's children into a dict; `parse_xml` walks the tree and stacks those dicts.

**How I get it.** `(child.text or "")` reads each leaf. The `or ""` is DEC-015: an empty XML
element gives `None`, while the JSON export gives `""` for the same absent value. Without it the
two sources are not comparable and every blank count reports a false difference.

**Why the others care.** `exceptions` is keyed on the **original tag**, not the lowered name, so
the JSON and XML exception tables are not interchangeable — Shawn will look for that in review.
And `warehouses` is returned but is *not* an output table: it must never become a column in the
six CSVs. It is kept because it carries lat/long, which is useful for an EDA distance figure.

In [ ]:
def element_to_record(element, exceptions=None):
    """One XML element -> one dict. Child tag becomes the column, child text becomes the value.

    `exceptions` maps an original tag (e.g. "Review_Text") straight to its final column name,
    bypassing the .lower() rule. Text is read as `child.text or ""` — see DEC-015.
    """
    exceptions = exceptions or {}
    return {exceptions.get(child.tag, child.tag.lower()): (child.text or "")
            for child in element}


def parse_xml(path, exceptions=None):
    """Read the operations XML export and return flat tables with source-native values.

    Returns
    -------
    tables : dict of str -> DataFrame   orders, order_items, deliveries, products,
                                        product_reviews, warehouses (not an output table)
    metadata : dict                     root attributes plus the Export_Metadata block
    """
    root = ET.parse(path).getroot()

    orders, order_items, deliveries = [], [], []
    for order in root.findall("Orders/Order"):
        orders.append(element_to_record(order.find("Header"), exceptions))
        deliveries.append(element_to_record(order.find("Delivery"), exceptions))
        order_items.extend(element_to_record(item, exceptions)
                           for item in order.findall("Shopping_Cart/Item"))

    tables = {
        "orders":          pd.DataFrame(orders),
        "order_items":     pd.DataFrame(order_items),
        "deliveries":      pd.DataFrame(deliveries),
        "products":        pd.DataFrame([element_to_record(p, exceptions)
                                         for p in root.findall("ProductCatalogue/Product")]),
        "product_reviews": pd.DataFrame([element_to_record(r, exceptions)
                                         for r in root.findall("ProductReviews/Review")]),
        "warehouses":      pd.DataFrame([element_to_record(w, exceptions)
                                         for w in root.findall("WarehouseDirectory/Warehouse")]),
    }
    for df in tables.values():
        df.insert(0, "source_system", "XML")

    metadata = dict(root.attrib)
    metadata.update(element_to_record(root.find("Export_Metadata")))
    return tables, metadata

#### Pass 1 — the general rule alone

**Why I need it.** DEC-012. Encoding the exceptions before running the rule would leave no
evidence of how they were found, and a reviewer could not tell a bug from a design choice.

**What it is.** `parse_xml` with no exceptions, checked against the data dictionary.

**How I get it.** The same `check_names` defined in §1.1 — reused, not rewritten, so the JSON
and XML sides are judged by identical criteria.

**Why the others care.** The right-hand list of this output is the derived-field inventory for
the XML side. Those are the `source_format = derived` rows Shawn and Jasmine own in the mapping.

In [ ]:
naive_xml, _ = parse_xml(XML_PATH)
check_names(naive_xml, DICT_PATH)

#### Reading pass 1, and the exceptions it justifies

Three tags convert cleanly but land on the wrong side of the raw/clean boundary, and **no tag
breaks the `.lower()` rule** — unlike the JSON side, where `prior12MOrders` did. That asymmetry
is itself a finding: the XML's explicit underscores leave nothing for a rule to guess wrong.

| Tag | `.lower()` gives | Dictionary wants | Why an exception |
|---|---|---|---|
| `Customer_Note` | `customer_note` | `customer_note_clean`, `promo_code` | one field, two targets |
| `Review_Text` | `review_text` | seven `product_reviews` fields | one field, seven targets |
| `Product_Description` | `product_description` | `product_description_clean` | one field, one cleaned target |

The left list is never renamed to a target — a source field that fans out to several targets has
no single correct target name. It is suffixed `_raw` to mark it as an *input* to derivation, and
the fan-out is recorded in the mapping CSV (DEC-014).

In [ ]:
# Written from the pass-1 report above, not before it. Keys are original tags.
XML_NAME_EXCEPTIONS = {
    "Customer_Note":       "customer_note_raw",
    "Review_Text":         "review_body_raw",
    "Product_Description": "product_description_raw",
}

#### Pass 2 — the corrected rule

**Why I need it.** To show the fix worked, and to leave the clean run as the notebook's evidence.

**What it is.** `xml_tables` — the frames every later section uses.

**How I get it.** Re-parse with the exception table, re-run the same check, drop the pass-1 copy.

**Why the others care.** After this cell the two sources have identical column names for shared
tables, which is the precondition for the overlap work in §1.3 and the reconciliation in §5.

In [ ]:
xml_tables, xml_meta = parse_xml(XML_PATH, XML_NAME_EXCEPTIONS)
check_names(xml_tables, DICT_PATH)
del naive_xml

print("\nExport metadata:", xml_meta)
for name, df in xml_tables.items():
    print(f"{name:16s} {df.shape[0]:,} rows x {df.shape[1]:>2} cols")

#### Profile

**Why I need it.** Task 1 asks for candidate keys, duplicate evidence and the date, boolean,
currency, percentage and missing-value formats. This is where each is shown rather than claimed.

**What it is.** Three checks: is the intended key unique, what is blank, and what do the values
literally look like.

**How I get it.** The same three cells as §1.1, run against `xml_tables` — deliberately identical
so §1.3 can put the two profiles side by side.

**Why the others care.** Every row of this output turns into work for someone: the duplicate
counts are Yandu's canonical-row problem (Q2), and each format below is a conversion Jasmine
writes and a `VAL-SCHEMA-` check Yandu runs.

In [ ]:
XML_KEYS = {
    "orders":          "order_id",
    "order_items":     "order_item_id",
    "deliveries":      "delivery_id",
    "products":        "product_id",
    "product_reviews": "review_id",
}

profile_xml = []
for name, key in XML_KEYS.items():
    df = xml_tables[name]
    profile_xml.append({
        "table":       name,
        "rows":        len(df),
        "key":         key,
        "unique_keys": df[key].nunique(),
        "duplicate_key_rows": len(df) - df[key].nunique(),
        "cols_with_blank": int((df == "").any().sum()),
    })

pd.DataFrame(profile_xml)

In [ ]:
# Missing-value convention. Everything is a string here, so a blank is "" and never None.
for name, df in xml_tables.items():
    blanks = (df == "").sum()
    blanks = blanks[blanks > 0]
    for col, n in blanks.items():
        print(f"{name}.{col:22s} {n:>6,} blank rows")

In [ ]:
# The formats themselves — the evidence behind every normalisation rule in §1.3.
print(xml_tables["orders"][["order_timestamp", "order_price", "delivery_charges",
                            "coupon_discount", "expedited_delivery"]].head(3).to_string(index=False))
print()
print(xml_tables["products"][["unit_price", "launch_date", "recyclable_packaging"]].head(3).to_string(index=False))
print()
print("Every XML column arrives as text:", xml_tables["orders"].dtypes.unique())

#### Hand-off note for §1.3

Two facts to carry forward, both of which the cells above evidence:

1. `order_items` row counts differ between the sources (JSON 8,826 vs XML 8,833) while `orders`
   and `product_reviews` counts match exactly. That asymmetry is the first thing §1.3 must
   explain — it is either genuine extra items or a duplicate pattern, and guessing is not an
   option.
2. Both sources carry duplicate business keys within themselves. Whether the duplicate rows are
   field-identical is a §1.3 question, and the answer decides how hard Q2 is for Yandu.